# Keras → TFLite, ONNX, CoreML Conversion

This notebook converts a fine-tuned Keras model into:
- TensorFlow Lite (Android)
- ONNX (cross-platform)
- CoreML (iOS)

It uses:
- final_model.keras
- labels.json (authoritative class mapping)

No manual labels. No overrides. Safe & reproducible.

In [ ]:
import json
import time
from pathlib import Path
import tensorflow as tf

print('TensorFlow:', tf.__version__)

In [ ]:
# ===== PATHS (HARDCODED AS REQUESTED) =====

BASE_DIR = Path('/Users/priyank.rastogi@zomato.com/projects/Gymie/frontend/ml-model')

MODEL_PATH = BASE_DIR / 'final_model.keras'
LABELS_PATH = BASE_DIR / 'labels.json'

assert MODEL_PATH.exists(), f'Model not found: {MODEL_PATH}'
assert LABELS_PATH.exists(), f'Labels not found: {LABELS_PATH}'

print('✅ Paths verified')

In [ ]:
# ===== LOAD MODEL =====

model = tf.keras.models.load_model(MODEL_PATH)
model.summary()

In [ ]:
# ===== LOAD LABELS =====

with open(LABELS_PATH, 'r') as f:
    class_names = json.load(f)

num_classes = len(class_names)
output_units = model.output_shape[-1]

assert num_classes == output_units, (
    f'Label count ({num_classes}) does not match model output ({output_units})'
)

print(f'✅ {num_classes} classes aligned correctly')

In [ ]:
# ===== UNIQUE OUTPUT NAMES =====

ts = int(time.time())

TFLITE_PATH = BASE_DIR / f'vision_{ts}.tflite'
ONNX_PATH = BASE_DIR / f'vision_{ts}.onnx'
COREML_PATH = BASE_DIR / f'vision_{ts}.mlpackage'

print('Outputs will be saved as:')
print(TFLITE_PATH.name)
print(ONNX_PATH.name)
print(COREML_PATH.name)

## Convert to TensorFlow Lite (Android)

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]

tflite_model = converter.convert()

with open(TFLITE_PATH, 'wb') as f:
    f.write(tflite_model)

print('✅ TFLite saved:', TFLITE_PATH)

## Convert to ONNX

In [ ]:
!pip install -q tf2onnx onnx

In [ ]:
import tf2onnx

spec = (tf.TensorSpec(model.input_shape, tf.float32, name='input'),)

tf2onnx.convert.from_keras(
    model,
    input_signature=spec,
    opset=13,
    output_path=str(ONNX_PATH)
)

print('✅ ONNX saved:', ONNX_PATH)

## Convert to CoreML (macOS only)

In [ ]:
!pip install -q coremltools==6.3 protobuf==3.20.3 numpy<2

In [ ]:
import coremltools as ct

mlmodel = ct.convert(
    model,
    source='tensorflow',
    inputs=[ct.ImageType(
        name='image',
        shape=(1, 224, 224, 3),
        scale=1/255.0
    )],
    classifier_config=ct.ClassifierConfig(class_names)
)

mlmodel.save(str(COREML_PATH))

print('✅ CoreML saved:', COREML_PATH)

## ✅ Conversion Complete